# register-buffer — worked example 2: Verify Buffer Values Survive a state_dict Roundtrip

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `register-buffer`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Because `register_buffer` tensors appear in `state_dict()`, they are saved and restored when you call `torch.save` / `load_state_dict`. This makes buffers suitable for non-learnable persistent state like attention masks or normalization statistics. The buffer value at save time is exactly what you get back at load time.

## Worked solution

**Step 1 — define a module with a buffer holding meaningful data.** We create a `TokenMask` module that stores a fixed attention mask as a registered buffer.

**Step 2 — mutate the buffer.** After construction, we modify `self.mask` directly (simulating an update). This is allowed — buffers are regular tensors, not locked.

**Step 3 — save state_dict.** `module.state_dict()` returns an `OrderedDict` that includes the buffer under its registered name.

**Step 4 — create a fresh module and load.** A new `TokenMask()` starts with its default buffer. We call `new_module.load_state_dict(sd)` to overwrite it with the saved values.

**Step 5 — verify.** The buffer in the new module matches the saved values exactly, confirming the roundtrip works.

In [ ]:
import torch as t
import torch.nn as nn

class TokenMask(nn.Module):
    def __init__(self, seq_len: int):
        super().__init__()
        # Causal attention mask: lower-triangular boolean
        mask = t.tril(t.ones(seq_len, seq_len, dtype=t.bool))
        self.register_buffer('mask', mask)

    def forward(self, attn_scores: t.Tensor) -> t.Tensor:
        return attn_scores.masked_fill(~self.mask, float('-inf'))

# --- exercise and print ---
t.manual_seed(0)
seq_len = 4
m1 = TokenMask(seq_len)
print('Original mask:')
print(m1.mask.int())

# Save state dict
sd = m1.state_dict()
print('\nstate_dict keys:', list(sd.keys()))  # ['mask']

# Load into a fresh module
m2 = TokenMask(seq_len)
m2.load_state_dict(sd)
print('\nLoaded mask matches:', t.equal(m1.mask, m2.mask))  # True
print('mask in buffers:', 'mask' in dict(m2.named_buffers()))  # True
print('mask in params: ', 'mask' in dict(m2.named_parameters()))  # False